# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Khuld13/ML-intern-starter/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
%cd /content
!rm -rf ML-intern-starter
!git clone https://github.com/Khuld13/ML-intern-starter.git
%cd ML-intern-starter

/content
Cloning into 'ML-intern-starter'...
remote: Enumerating objects: 211, done.
remote: Counting objects: 100% (211/211), done.
remote: Compressing objects: 100% (164/164), done.
remote: Total 211 (delta 106), reused 88 (delta 31), pack-reused 0 (from 0)
Receiving objects: 100% (211/211), 1.92 MiB | 6.64 MiB/s, done.
Resolving deltas: 100% (106/106), done.
/content/ML-intern-starter


In [4]:
import duckdb
from google.colab import userdata

con = duckdb.connect()
con.sql("INSTALL httpfs; LOAD httpfs;")

hf_token = userdata.get('HF_TOKEN')
con.sql(f"""
CREATE OR REPLACE SECRET hf_secret (
    TYPE HUGGINGFACE,
    TOKEN '{hf_token}'
);
""")

REL = 'hf://datasets/FlyRank/internship-warehouse'

con.sql(f"""
CREATE OR REPLACE VIEW dim_content AS
SELECT * FROM read_parquet('{REL}/dim_content.parquet');
""")

con.sql(f"""
CREATE OR REPLACE VIEW fact_daily AS
SELECT * FROM read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')
WHERE month = '2026-03';
""")

print("Connected. Views ready.")

Connected. Views ready.


In [5]:
con.sql("""
    CREATE OR REPLACE VIEW fact_march AS
    SELECT * FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet')
""")

con.sql("""
    CREATE OR REPLACE VIEW fact_feb AS
    SELECT * FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-02/data_0.parquet')
""")

print("fact_march and fact_feb ready.")

fact_march and fact_feb ready.


In [6]:
monthly_compare = con.sql("""
    WITH march_agg AS (
        SELECT content_hash_id, SUM(gsc_impressions) AS impressions_march
        FROM fact_march
        WHERE gsc_data_available = TRUE
        GROUP BY content_hash_id
    ),
    feb_agg AS (
        SELECT content_hash_id, SUM(gsc_impressions) AS impressions_feb
        FROM fact_feb
        WHERE gsc_data_available = TRUE
        GROUP BY content_hash_id
    )
    SELECT
        m.content_hash_id,
        f.impressions_feb,
        m.impressions_march,
        CASE WHEN m.impressions_march < f.impressions_feb THEN 1 ELSE 0 END AS declined_flag
    FROM march_agg m
    JOIN feb_agg f USING (content_hash_id)
""").df()

# ML-07's volume-floor rule (Signal 2, CONFIRMED)
pop = monthly_compare[monthly_compare['impressions_march'] >= 250].copy()

# Bring in content-level features to model with
dim = con.sql("SELECT * FROM dim_content").df()
df = pop.merge(dim, on='content_hash_id', how='left')

# Exclude: the label itself, the two raw inputs that DEFINE the label,
# and the known-leaky/known-invalid columns from ML-06
leakage_cols = [
    'declined_flag', 'impressions_feb', 'impressions_march',   # label + its direct inputs
    'trend_pct', 'trend_direction', 'is_declining_label',       # ML-06: same fact, 3 forms
    'days_since_update',                                        # ML-06: structurally invalid (July snapshot)
]
candidate_features = [c for c in df.columns if c not in leakage_cols]

print("Rows after volume filter (impressions_march >= 250):", len(df))
print("\nLabel balance (declined_flag):")
print(df['declined_flag'].value_counts(normalize=True))
print("\nCandidate feature count:", len(candidate_features))
print(candidate_features)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows after volume filter (impressions_march >= 250): 68581

Label balance (declined_flag):
declined_flag
0    0.771759
1    0.228241
Name: proportion, dtype: float64

Candidate feature count: 26
['content_hash_id', 'client_hash_id', 'keyword_hash_id', 'url_hash_id', 'keyword_char_count', 'keyword_token_count', 'url_char_count', 'content_created_date', 'content_updated_date', 'content_type', 'search_volume', 'competition', 'competition_level', 'cpc', 'main_intent', 'backlinks', 'category_count', 'keyword_created_date', 'provider_used', 'model_used', 'char_count', 'word_count', 'last_optimized_date', 'optimization_eligible_date', 'is_published', 'is_deleted']


In [7]:
from sklearn.model_selection import GroupShuffleSplit

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(df, groups=df['client_hash_id']))

train_df = df.iloc[train_idx]
test_df = df.iloc[test_idx]

print("Train rows:", len(train_df), " Test rows:", len(test_df))
print("Train clients:", train_df['client_hash_id'].nunique(), " Test clients:", test_df['client_hash_id'].nunique())
print("\nTrain label balance:\n", train_df['declined_flag'].value_counts(normalize=True))
print("\nTest label balance:\n", test_df['declined_flag'].value_counts(normalize=True))

Train rows: 54873  Test rows: 13708
Train clients: 27  Test clients: 7

Train label balance:
 declined_flag
0    0.787036
1    0.212964
Name: proportion, dtype: float64

Test label balance:
 declined_flag
0    0.710607
1    0.289393
Name: proportion, dtype: float64


In [9]:
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer

# df = your w05 filtered table, candidate_features = your w05 26-column list
X = df[candidate_features]
y = df['declined_flag']
groups = df['client_hash_id']

# split feature list by dtype, same as your w05 approach
numeric_features = X.select_dtypes(include=['int64', 'float64', 'Int64']).columns.tolist()
categorical_features = X.select_dtypes(include=['object', 'bool']).columns.tolist()
# drop id/key columns that shouldn't be modeled as features
id_cols = ['content_hash_id', 'client_hash_id', 'keyword_hash_id', 'url_hash_id']
numeric_features = [c for c in numeric_features if c not in id_cols]
categorical_features = [c for c in categorical_features if c not in id_cols]

preprocessor = ColumnTransformer([
    ('num', Pipeline([
        ('impute', SimpleImputer(strategy='median')),
        ('scale', StandardScaler())
    ]), numeric_features),
    ('cat', Pipeline([
        ('impute', SimpleImputer(strategy='constant', fill_value='missing')),
        ('onehot', OneHotEncoder(handle_unknown='ignore'))
    ]), categorical_features)
])

def make_model():
    return Pipeline([
        ('prep', preprocessor),
        ('clf', LogisticRegression(class_weight='balanced', max_iter=1000))
    ])

# --- BEFORE: naive random split, ignores client grouping ---
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
naive_model = make_model().fit(X_train, y_train)
naive_auc = roc_auc_score(y_test, naive_model.predict_proba(X_test)[:, 1])

# --- AFTER: client-grouped holdout, what you actually used in w05 ---
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=groups))
grouped_model = make_model().fit(X.iloc[train_idx], y.iloc[train_idx])
grouped_auc = roc_auc_score(y.iloc[test_idx], grouped_model.predict_proba(X.iloc[test_idx])[:, 1])

print(f"Naive random split AUC:   {naive_auc:.3f}")
print(f"Client-grouped split AUC: {grouped_auc:.3f}")

Naive random split AUC:   0.643
Client-grouped split AUC: 0.544


In [11]:
import numpy as np

naive_aucs, grouped_aucs = [], []

for seed in [0, 1, 7, 42, 99]:
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=seed, stratify=y
    )
    m = make_model().fit(X_train, y_train)
    naive_aucs.append(roc_auc_score(y_test, m.predict_proba(X_test)[:, 1]))

    gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=seed)
    tr_idx, te_idx = next(gss.split(X, y, groups=groups))
    m2 = make_model().fit(X.iloc[tr_idx], y.iloc[tr_idx])
    grouped_aucs.append(roc_auc_score(y.iloc[te_idx], m2.predict_proba(X.iloc[te_idx])[:, 1]))

print(f"Naive:   mean={np.mean(naive_aucs):.3f}  std={np.std(naive_aucs):.3f}  runs={[round(a,3) for a in naive_aucs]}")
print(f"Grouped: mean={np.mean(grouped_aucs):.3f}  std={np.std(grouped_aucs):.3f}  runs={[round(a,3) for a in grouped_aucs]}")

Naive:   mean=0.650  std=0.004  runs=[np.float64(0.65), np.float64(0.656), np.float64(0.649), np.float64(0.643), np.float64(0.652)]
Grouped: mean=0.551  std=0.019  runs=[np.float64(0.547), np.float64(0.538), np.float64(0.538), np.float64(0.544), np.float64(0.588)]


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

Under a naive random row-level split, the logistic regression model measured an average AUC of 0.650 (std 0.004 across 5 seeds). Under a client-grouped holdout — where all pages from a given client are kept entirely in either train or test — the same model measured an average AUC of 0.551 (std 0.019 across the same 5 seeds). This ~0.10 AUC gap is consistent across seeds and is directionally consistent with client-level leakage: in the naive split, pages from the same client appear in both train and test, letting the model partially memorize client-specific patterns rather than learn signal that transfers to unseen clients. The client-grouped result — close to but modestly above chance (0.5) — is the more honest estimate of how this model would perform on a client it has never seen, and suggests the current feature set has only weak observed ability to generalize across clients under this label and filter definition.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.